# Pipeline Orchestration Mock Test

This notebook models the document lifecycle processing. We use mocked extractors and agents to validate the folder creation, state transitions, and manifest integrity before integrating the heavy ML components.

In [1]:
# Enable autoreload to automatically pick up changes in local modules
%load_ext autoreload
%autoreload 2

In [2]:
import logging
import time
from pathlib import Path

# Import core storage and manifest managers
from doc_agent.infrastructure.storage import LocalStorageManager
from doc_agent.infrastructure.manifest_manager import LocalManifestManager
from doc_agent.schemas.manifest import PageStatus
from doc_agent.utils.logger import setup_logger

# Configure logging
logger = setup_logger(
    name="005_orchestrator_test", 
    level=logging.INFO,
    log_file="notebook_experiments.log",
    log_dir=Path.cwd() / "logs"
)

# Define the isolated workspace directory for this mock run
notebook_root = Path.cwd()
workspace_dir = notebook_root / "data" / "02_interim" / "test_chapter_workspace"
logger.info(f"Workspace directory set to: {workspace_dir.relative_to(notebook_root)}")

2026-05-09 12:36:35 |     INFO | 005_orchestrator_test:3797538887.py:22 - Workspace directory set to: data/02_interim/test_chapter_workspace


## Mocks (Stubs) for Heavy Operations

Instead of calling PyPDFium2, Docling, and the NanoGPT API, we simulate their behavior and execution time. This allows us to test the pipeline logic and state machine instantly without spending API tokens or waiting for GPU inference.

In [3]:
def mock_slice_pdf(pdf_path: Path) -> list[str]:
    """
    Simulates slicing a PDF into multiple pages.
    
    Args:
        pdf_path (Path): Path to the source PDF.
        
    Returns:
        list[str]: A list of generated page IDs.
    """
    logger.info(f"Slicing PDF '{pdf_path.name}' into pages...")
    time.sleep(1)  # Simulate I/O latency
    return ["page_001", "page_002", "page_003"]

def mock_docling_parser(page_id: str) -> str:
    """
    Simulates the Docling AST parsing and XML tagging process.
    """
    logger.info(f"Docling is parsing layout for {page_id}...")
    time.sleep(1)  # Simulate parsing latency
    return f"<text_1>Raw extracted content from {page_id}</text_1>"

def mock_vlm_agent(tagged_text: str) -> str:
    """
    Simulates the LLM healing process (Semantic Normalization).
    """
    logger.info(" VLM Agent is healing the markdown structure...")
    time.sleep(1)  # Simulate API latency
    
    # Strip tags to simulate a "cleaned" output
    clean_text = tagged_text.replace("<text_1>", "").replace("</text_1>", "")
    return f"{clean_text}\n\n> This page was successfully healed by the mock VLM Agent."

## The Dummy Orchestrator

This class represents the "Factory Floor Manager". It directs the flow of data between the functional modules (extractors and agents) and safely records state transitions in the `manifest.json`.

In [4]:
class DummyOrchestrator:
    """
    A mock implementation of the DocumentProcessor to test the workflow logic.
    """
    def __init__(self, workspace_dir: Path, manifest: LocalManifestManager):
        self.workspace = workspace_dir
        self.manifest = manifest

    def process(self, pdf_path: Path):
        """Executes the simulated ingestion pipeline on a target PDF."""
        logger.info(f"STARTING pipeline for: {pdf_path.name}")
        
        # Step 1: Document slicing and manifest initialization
        page_ids = mock_slice_pdf(pdf_path)
        self.manifest.init_manifest(str(pdf_path), total_pages=len(page_ids))

        # Step 2: Page-by-page processing loop
        for page_id in page_ids:
            logger.info(f"--- Processing {page_id} ---")
            
            # 2.1: Physical layout parsing
            tagged_md = mock_docling_parser(page_id)
            self.manifest.update_page_status(page_id, PageStatus.TAGGED)
            
            # 2.2: Semantic healing
            clean_md = mock_vlm_agent(tagged_md)
            
            # 2.3: Persist the healed artifact to disk
            output_dir = self.workspace / "05_md_clean"
            output_dir.mkdir(parents=True, exist_ok=True)
            
            output_file = output_dir / f"{page_id}.md"
            output_file.write_text(clean_md, encoding="utf-8")
            
            # 2.4: Update manifest to reflect completion
            self.manifest.update_page_status(page_id, PageStatus.CLEANED)
            logger.info(f" {page_id} successfully saved to {output_file.relative_to(Path.cwd())}")

        logger.info("Pipeline execution completed successfully!")

## Execution

Instantiate the core storage infrastructure and run the dummy orchestrator on a fake PDF path.

In [5]:
# 1. Initialize infrastructure (Storage and Manifest Manager)
storage = LocalStorageManager(base_dir=workspace_dir)
manifest_manager = LocalManifestManager(storage=storage, doc_id="dummy_chapter_1")

# 2. Initialize and run the Orchestrator
orchestrator = DummyOrchestrator(workspace_dir, manifest_manager)

# Use a fake path for the simulation
dummy_pdf_path = Path("data/01_raw/dummy_pue_chapter.pdf")

# Execute the pipeline!
orchestrator.process(dummy_pdf_path)

2026-05-09 12:41:04 |     INFO | 005_orchestrator_test:333966360.py:11 - STARTING pipeline for: dummy_pue_chapter.pdf
2026-05-09 12:41:04 |     INFO | 005_orchestrator_test:1618292714.py:11 - Slicing PDF 'dummy_pue_chapter.pdf' into pages...
2026-05-09 12:41:05 |     INFO | 005_orchestrator_test:333966360.py:19 - --- Processing page_001 ---
2026-05-09 12:41:05 |     INFO | 005_orchestrator_test:1618292714.py:19 - Docling is parsing layout for page_001...
2026-05-09 12:41:06 |     INFO | 005_orchestrator_test:1618292714.py:27 -  VLM Agent is healing the markdown structure...
2026-05-09 12:41:07 |     INFO | 005_orchestrator_test:333966360.py:37 -  page_001 successfully saved to data/02_interim/test_chapter_workspace/05_md_clean/page_001.md
2026-05-09 12:41:07 |     INFO | 005_orchestrator_test:333966360.py:19 - --- Processing page_002 ---
2026-05-09 12:41:07 |     INFO | 005_orchestrator_test:1618292714.py:19 - Docling is parsing layout for page_002...
2026-05-09 12:41:08 |     INFO | 0